In [ ]:
# Libraries needed for NLP and machine learning
# - nltk: tokenization and stemming
# - tensorflow / keras: model building and training
# - numpy / random / json: utility operations
import nltk
nltk.download('punkt_tab')
from nltk.stem.lancaster import LancasterStemmer

# Stemmer used to reduce words to their root form
stemmer = LancasterStemmer()

import tensorflow as tf
import numpy as np
import random
import json

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\shahd\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
# Load intents JSON file which contains example patterns and responses
file_path = r"intents.json"

with open(file_path, "r", encoding="utf-8") as file:
    intents = json.load(file)

In [ ]:
# Quick display of loaded intents (for debugging)
intents

{'intents': [{'context': [''],
   'patterns': ['Hi there',
    'Greetings',
    'Is anyone there?',
    'Hey',
    'Hola',
    'Hello',
    'Good day'],
   'responses': ['Hi there!',
    'Good to see you again',
    'Hi there, how can I help?'],
   'tag': 'greeting'},
  {'context': [''],
   'patterns': ['How are you',
    "What's up?",
    'How is it going?',
    'How are you today?'],
   'responses': ['Hello, thanks for asking',
    'Good to see you again',
    'Hi there, how can I help?'],
   'tag': 'how_are_you'},
  {'context': [''],
   'patterns': ['Who are you',
    "What's your name?",
    'What are you',
    'Who is this'],
   'responses': ["I'm Travel Buddy",
    'My name is Travel Buddy',
    "Thanks for asking, I'm Travel Buddy!"],
   'tag': 'who_are_you'},
  {'context': [''],
   'patterns': ['Bye',
    'See you later',
    'Goodbye',
    'Nice chatting to you, bye',
    'Till next time'],
   'responses': ['See you!', 'Have a nice day', 'Bye! Come back again soon.'],
   'tag'

In [ ]:
# Prepare lists for words, classes (tags) and document tuples
words = []
classes = []
documents = []
ignore = ['?']  # characters to ignore when processing

# Iterate through each intent and tokenize the patterns
for intent in intents['intents']:
  for pattern in intent['patterns']:
    # Tokenize each pattern (sentence) into words
    w = nltk.word_tokenize(pattern)
    words.extend(w)  # add to overall vocabulary list
    documents.append((w, intent['tag']))  # pair of token list and its tag
    if intent['tag'] not in classes:
      classes.append(intent['tag'])  # unique classes (labels)

In [ ]:
# Stem and lower-case all words, remove duplicates and sort
words = [stemmer.stem(w.lower()) for w in words if w not in ignore]
words = sorted(list(set(words)))
classes = sorted(list(set(classes)))

# Print some basic stats to confirm processing
print(len(documents), "documents")
print(len(classes), "classes", classes)
print(len(words), "uniques stemmed words", words)

35 documents
7 classes ['goodbye', 'greeting', 'how_are_you', 'options', 'thanks', 'travel_tips', 'who_are_you']
57 uniques stemmed words ["'s", ',', 'a', 'anyon', 'ar', 'awesom', 'be', 'bye', 'can', 'chat', 'could', 'day', 'do', 'for', 'get', 'giv', 'going', 'good', 'goodby', 'greet', 'hello', 'help', 'hey', 'hi', 'hol', 'how', 'i', 'is', 'it', 'lat', 'me', 'mod', 'nam', 'next', 'nic', 'off', 'op', 'provid', 'see', 'support', 'thank', 'that', 'ther', 'thi', 'til', 'tim', 'tip', 'to', 'today', 'travel', 'up', 'want', 'what', 'who', 'with', 'yo', 'you']


In [ ]:
training = []
output = []

output_empty = [0] * len(classes)
for doc in documents:

  bag = []
  pattern_words = doc[0]
  # Stem each word in the pattern
  pattern_words = [stemmer.stem(word.lower()) for word in pattern_words]
  # Create bag of words: 1 if word exists in the pattern, else 0
  for w in words:
    bag.append(1) if w in pattern_words else bag.append(0)

  # Create the output row (one-hot for the class/tag)
  output_row = list(output_empty)
  output_row[classes.index(doc[1])] = 1

  training.append([bag, output_row])

# Shuffle and convert to numpy arrays ready for training
random.shuffle(training)
training = np.array(training, dtype=object)

train_x = list(training[:,0])
train_y = list(training[:,1])

In [ ]:
# Build a simple feed-forward neural network with Keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD

# Ensure training data are float arrays for Keras
train_x = np.array(train_x, dtype=np.float32)
train_y = np.array(train_y, dtype=np.float32)

# Model architecture: two hidden layers and a softmax output
model = Sequential([
    Dense(10, activation='relu', input_shape=(len(train_x[0]),)),
    Dense(10, activation='relu'),
    Dense(len(train_y[0]), activation='softmax')
])

# Compile with SGD optimizer and categorical crossentropy loss
model.compile(
    optimizer=SGD(learning_rate=0.01, momentum=0.9),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train the model
model.fit(
    train_x,
    train_y,
    epochs=1000,
    batch_size=8,
    verbose=1
)

# Save trained model to disk
model.save("model.keras")


Epoch 1/1000


5/5 [==============================] - 1s 4ms/step - loss: 1.9272 - accuracy: 0.3429
Epoch 2/1000
5/5 [==============================] - 0s 3ms/step - loss: 1.9218 - accuracy: 0.2857
Epoch 3/1000
5/5 [==============================] - 0s 3ms/step - loss: 1.9051 - accuracy: 0.3429
Epoch 4/1000
5/5 [==============================] - 0s 3ms/step - loss: 1.8921 - accuracy: 0.3714
Epoch 5/1000
5/5 [==============================] - 0s 3ms/step - loss: 1.8756 - accuracy: 0.4000
Epoch 6/1000
5/5 [==============================] - 0s 3ms/step - loss: 1.8605 - accuracy: 0.3429
Epoch 7/1000
5/5 [==============================] - 0s 3ms/step - loss: 1.8437 - accuracy: 0.4000
Epoch 8/1000
5/5 [==============================] - 0s 3ms/step - loss: 1.8277 - accuracy: 0.4000
Epoch 9/1000
5/5 [==============================] - 0s 2ms/step - loss: 1.8036 - accuracy: 0.4571
Epoch 10/1000
5/5 [==============================] - 0s 2ms/step - loss: 1.7800 - accuracy: 0.4571
Epoch 11/1000
5/

In [ ]:
# Save processed training data for later use (words, classes, train_x, train_y)
import pickle
pickle.dump( { 'words': words, 'classes':classes, 'train_x':train_x, 'train_y':train_y}, open("training_data", "wb"))

In [ ]:
# Load previously saved training data (if available) to avoid retraining
data = pickle.load( open("training_data", "rb"))
words = data['words']
classes = data['classes']
train_x = data['train_x']
train_y = data['train_y']

In [ ]:
# Ensure intents are loaded (used by the response function below)
with open('intents.json') as json_data:
  intents = json.load(json_data)

In [ ]:
# Load the trained Keras model from disk for inference
from tensorflow.keras.models import load_model

model = load_model("model.keras")

In [ ]:
# Helper: tokenize, stem and return list of words from a sentence
def clean_up_sentence(sentence):

  # Tokenize the sentence into words
  sentence_words = nltk.word_tokenize(sentence)

  # Stem and lower-case each word for consistency with training
  sentence_words = [stemmer.stem(word.lower()) for word in sentence_words]
  return sentence_words

# Create a bag-of-words array for the sentence based on the known vocabulary
def bow(sentence, words, show_details=False):
  sentence_words = clean_up_sentence(sentence)
  bag = [0]*len(words)
  for s in sentence_words:
        for i,w in enumerate(words):
            if w == s:
                bag[i] = 1
                if show_details:
                    print ("found in bag: %s" % w)
  return(np.array(bag))

In [ ]:
# Classification threshold for model confidence
ERROR_THRESHOLD = 0.30
def classify(sentence):

    # Predict probabilities for each class and filter by threshold
    results = model.predict(
    np.array([bow(sentence, words)], dtype=np.float32),
    verbose=0)[0]
    results = [[i,r] for i,r in enumerate(results) if r>ERROR_THRESHOLD]
    results.sort(key=lambda x: x[1], reverse=True)  # sort by probability
    return_list = []
    for r in results:
        # append tuple (class_label, probability)
        return_list.append((classes[r[0]], r[1]))
    return return_list

# Select an appropriate response based on the classified intent
def response(sentence, userID='123', show_details=False):
    results = classify(sentence)
    if results:
        while results:
            for i in intents['intents']:
                if i['tag'] == results[0][0]:
                    return print(random.choice(i['responses']))

            results.pop(0)

In [ ]:
# Quick smoke test: call response with empty input
response('')

Hi there, how can I help?


In [ ]:
# Example queries to test the chatbot
response('How are you')

Good to see you again


In [16]:
response('any tips for my trip')

Of course!


In [17]:
response('tips')

Any time!


In [18]:
response('travel tips')

Here's a tip: always pack light!


In [19]:
response('')

Hi there!


In [20]:
response('open travel tips')

Here's a tip: When traveling to a new city, be on the lookout for free walking tours!


In [21]:
response('travel tips')

Here's a tip: remember to make extra copies of your passport and other documents.


In [22]:
response('Available packages')

Good to see you again


In [23]:
response('Thanks')

Happy to help!


In [24]:
response('How are you')

Hello, thanks for asking


In [25]:
response('how are you doing')

Hi there, how can I help?


In [26]:
response('travel tips')

Always pack extra socks!


In [27]:
response('what are you doing')

Thanks for asking, I'm Travel Buddy!


In [28]:
response('')

Hi there, how can I help?


In [29]:
response('travel tips')

Always pack extra socks!


In [30]:
response('hi')

Good to see you again


In [31]:
response('hello there')

Hi there!
